In [4]:

# Data Preparation 

# STEP 1: We Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# optional: remove future warning
pd.set_option('future.no_silent_downcasting', True)

# STEP 2: Load the Dataset

df = pd.read_csv("DatasetDiabetes.csv")

# Rename target column to 'Class' to unify naming
if 'CLASS' in df.columns:
    df.rename(columns={'CLASS': 'Class'}, inplace=True)

print(" Dataset loaded successfully!\n")

# STEP 3: Basic Info
print("Dataset Info:")
print(df.info())
print("\nMissing Values per Column:")
print(df.isnull().sum())


# STEP 4: Data Cleaning (Safe & Robust)

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

# Fill numeric columns with median (robust to outliers)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical columns with mode (most frequent value)
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Remove duplicate rows if any
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()

print("\n Missing values handled and duplicates removed.")



# STEP 5: Encode Categorical Columns


# Convert Gender column from text to numeric if exists
if 'Gender' in df.columns:
    df['Gender'] = (
        df['Gender']
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({'M': 1, 'F': 0, 'MALE': 1, 'FEMALE': 0})
    )

# Normalize target column values
df['Class'] = (
    df['Class']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'N': 'Non-Diabetic',
        'NONDIABETIC': 'Non-Diabetic',
        'NON_DIABETIC': 'Non-Diabetic',
        'Y': 'Diabetic',
        'D': 'Diabetic',
        'DIABETIC': 'Diabetic',
        'P': 'Predict-Diabetic',
        'PREDICTDIABETIC': 'Predict-Diabetic',
        'PREDICT-DIABETIC': 'Predict-Diabetic',
    })
)

print("\n Categorical columns encoded successfully.")




# STEP 6: Split the Dataset


# Separate features (X) and target (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Check if stratification is possible (enough samples per class)
def can_stratify(series):
    counts = series.value_counts()
    return (len(counts) >= 2) and (counts.min() >= 2)

# Try stratified split first; fallback to normal split if needed
if can_stratify(y):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
else:
    print("\nStratified split not possible (too few samples in one class), We using random split instead.")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, shuffle=True
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, shuffle=True
    )

print("\n✅ Data Split Summary:")
print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")


# STEP 7: Save the Processed Files 


X_train.to_csv("X_train.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\n🎯 Files ready for model training:")
print("- X_train.csv")
print("- y_train.csv")
print("- X_test.csv")
print("- y_test.csv")
print("\n Data preprocessing completed successfully!")




 Dataset loaded successfully!

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   No_Pation  1000 non-null   int64  
 2   Gender     1000 non-null   object 
 3   AGE        1000 non-null   int64  
 4   Urea       1000 non-null   float64
 5   Cr         1000 non-null   int64  
 6   HbA1c      1000 non-null   float64
 7   Chol       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  Class      1000 non-null   object 
dtypes: float64(8), int64(4), object(2)
memory usage: 109.5+ KB
None

Missing Values per Column:
ID           0
No_Pation    0
Gender       0
AGE          0
Urea         0
Cr           0
HbA1c        0
Chol   